In [1]:
"""
simulate_guv.py
Matches the DGP in simulate_guv.f90 exactly.
- AR(1) is advanced at every age including age 25 (h=1), same as OBJECTIVE.f90
- t = h/10 where h=1..36 (age 25 = h=1, t=0.1)
- No Jensen correction, no wage index, no PCE deflator
Output: simulated_guv_fortran.csv with columns inc_25,...,inc_60
"""

import numpy as np
import pandas as pd

# ----------------------------------------------------------------
# PARAMETERS (hardcoded from f_guv_model.py / a01_parameters.py)
# ----------------------------------------------------------------

nsim = 50000
hmax = 36   # ages 25 to 60

# Life-cycle profile
a0 =  2.580861694
a1 =  0.811530031
a2 = -0.185093302

# HIP
sigma_alpha = 0.299819619
sigma_beta  = 0.196328895
corr_ab     = 0.767749193

# AR(1) mixture
pdf_ar  =  0.406559194
mu_eta1 = -0.085236482
sd_eta1 =  0.363928061
sd_eta2 =  0.06891405
sd_z0   =  0.713648178
rho1    =  0.959229453

# Nonemployment shock
nu_const = -3.352949544
nu_age   = -0.859498283
nu_z     = -5.034075647
nu_inter = -2.895204912
nu_lam   =  0.000265509

# Transitory shocks
pr_eps  =  0.129904327
mu_eps1 =  0.271112226
sd_eps1 =  0.284541004
sd_eps2 =  0.036545913

# ----------------------------------------------------------------
# DERIVED PARAMETERS
# ----------------------------------------------------------------
mu_eta2 = -mu_eta1 * pdf_ar / (1.0 - pdf_ar)
mu_eps2 = -mu_eps1 * pr_eps  / (1.0 - pr_eps)

# Cholesky of HIP covariance [[sigma_alpha^2, cov_ab],[cov_ab, sigma_beta^2]]
cov_ab = corr_ab * sigma_alpha * sigma_beta
L11 = sigma_alpha
L21 = cov_ab / sigma_alpha
L22 = np.sqrt(max(sigma_beta**2 - L21**2, 0.0))

# ----------------------------------------------------------------
# RNG (seeded for reproducibility)
# ----------------------------------------------------------------
rng = np.random.default_rng(42)

# ----------------------------------------------------------------
# HIP: draw alpha and beta (fixed per person across all ages)
# ----------------------------------------------------------------
rn_hip1 = rng.standard_normal(nsim)
rn_hip2 = rng.standard_normal(nsim)

alpha = L11 * rn_hip1
beta  = L21 * rn_hip1 + L22 * rn_hip2

# ----------------------------------------------------------------
# Initialize AR(1): z = sd_z0 * N(0,1)
# ----------------------------------------------------------------
rn_z0 = rng.standard_normal(nsim)
ar_z1 = sd_z0 * rn_z0

# ----------------------------------------------------------------
# MAIN SIMULATION LOOP (matches DO h=1,hmax in Fortran)
# ----------------------------------------------------------------
ysim = np.zeros((nsim, hmax))

for h in range(1, hmax + 1):
    age_s = h / 10.0   # t = h/10, same as Fortran

    # Draw random numbers for this age
    rn_p_ar  = rng.uniform(size=nsim)
    rn_eta   = rng.standard_normal(nsim)
    rn_unemp = rng.uniform(size=nsim)
    rn_nu    = rng.uniform(size=nsim)
    rn_p_eps = rng.uniform(size=nsim)
    rn_eps   = rng.standard_normal(nsim)

    # Advance AR(1) -- every period including h=1
    mask_ar = rn_p_ar <= pdf_ar
    ar_z1 = np.where(
        mask_ar,
        rho1 * ar_z1 + mu_eta1 + sd_eta1 * rn_eta,
        rho1 * ar_z1 + mu_eta2 + sd_eta2 * rn_eta
    )

    # Nonemployment shock
    xi    = nu_const + nu_age*age_s + nu_z*ar_z1 + nu_inter*age_s*ar_z1
    pnu   = np.exp(xi) / (1.0 + np.exp(xi))
    nu    = np.where(
        rn_unemp <= pnu,
        np.minimum(-np.log(np.maximum(rn_nu, 1e-15)) / nu_lam, 1.0),
        0.0
    )

    # Transitory shock: mixture of two normals
    eps = np.where(
        rn_p_eps <= pr_eps,
        mu_eps1 + sd_eps1 * rn_eps,
        mu_eps2 + sd_eps2 * rn_eps
    )

    # Income: (1-nu) * exp(g(t) + alpha + beta*t + z + eps)
    log_y = (a0 + a1*age_s + a2*age_s**2
             + alpha + beta*age_s
             + ar_z1 + eps)
    ysim[:, h-1] = np.maximum(0.0, (1.0 - nu) * np.exp(log_y))

# ----------------------------------------------------------------
# WRITE CSV
# ----------------------------------------------------------------
cols = [f'inc_{age}' for age in range(25, 61)]
df = pd.DataFrame(ysim, columns=cols)
df.to_csv(r'C:\Users\adv23\Dropbox\John-Jackson-Steve 2023\earning_dynamics\data\intermediate\simulated_guv_data.csv', index=False)
print('Done. Output: simulated_guv_fortran.csv')

Done. Output: simulated_guv_fortran.csv
